# Experiments — *What Happens Next?* video classification

Lab notebook for tracking architectures, runs, and results on the CSC_43M04_EP challenge (33-class Something-Something-v2-style data, 45,132 train / 6,746 val videos, 4 frames per video on disk).

## Modular architecture

All experiments here use a 3-slot modular video model (`models/modular.py`):

```
video  (B, T, 3, H, W)
  └─ SpatialEncoder    (B, T, d)    # per-frame, independent
  └─ TemporalProcessor (B, d')      # mixes across T
  └─ Classifier        (B, K)       # final logits
```

Each slot is swappable. Current registries:

| Slot | Options |
|---|---|
| Spatial | `resnet` (r18/r34/r50, pretrained opt.), `vit` (ti_16, s_16, b_16, b_32, l_16) |
| Temporal | `mean_pool`, `lstm`, `transformer` |
| Classifier | `linear`, `mlp` |

A run is defined by a Hydra config under `src/configs/experiment/<name>.yaml` that picks one option per slot + hyper-params. `build_model` checks shape compatibility between slots at construction time.

## Experiment 1 — ViT-S/16 + Transformer + MLP, from scratch

First run of the modular pipeline. Everything trained from random init (no ImageNet weights). Purpose: establish a from-scratch floor for a ViT-based architecture on this data scale, and validate the modular pipeline end-to-end.

**Config:** `experiment=vit_transformer_mlp` (`src/configs/experiment/vit_transformer_mlp.yaml`)

### Components

| Slot | Module | Config |
|---|---|---|
| Spatial | `ViTEncoder(variant=vit_s_16, pretrained=False)` | patch 16, 12 blocks, 6 heads, hidden 384, mlp 1536, 224² input → CLS → (B, T, 384) |
| Temporal | `TransformerTemporal(in_dim=384, num_heads=8, num_layers=2, dropout=0.1, max_len=64)` | prepend learnable CLS, learnable pos-embed, pre-norm TF blocks, take CLS → (B, 384) |
| Classifier | `MLPClassifier(in_dim=384, hidden_dim=512, num_classes=33, dropout=0.5)` | Linear→GELU→Dropout→Linear |

### Parameter budget (approx.)

| Module | Params |
|---|---|
| ViT-S/16 | ~22.0 M |
| Temporal transformer (2 layers, d=384) | ~1.8 M |
| MLP head (384→512→33) | ~0.2 M |
| **Total** | **~24.0 M** |

### Training setup

- **Data:** `num_frames=4`, `val_ratio=0.2` (internal split from `train/`), ImageNet normalization off (trained from scratch).
- **Optim:** Adam, `lr=1e-4`, no weight decay, no scheduler.
- **Batch:** `batch_size=8`, FP32, `num_workers=2`.
- **Epochs:** 20 (target ≈ 5 h wall-clock).
- **Hardware:** RTX 2000 Ada, 16 GB, 70 W TGP.
- **Checkpoint:** `/Data/challenge_sb/best_model_vit_s_scratch.pt` (best-by-val-acc).
- **Log:** `/Data/challenge_sb/logs/vit_s_scratch_20260423_185848.log`.
- **Started:** 2026-04-23 18:58.

### Hypothesis

From-scratch ViT at this data scale (45k videos ≈ 1/7 ImageNet) is expected to plateau well below a pretrained-CNN baseline. Realistic ceiling: **15–25 % val top-1**. The run is primarily a floor measurement and a pipeline sanity check — not a leaderboard contender.

### Results — failed

**Val top-1 plateaued at ~7 %** (random-chance floor is 1/33 ≈ 3 %). Well below the 15–25 % hypothesis, and 24 pts under Exp 2 (ResNet18, same T / lr / optim → 31.5 %).

#### Diagnosis

- **ViT-scratch needs data we don't have.** 45k clips is ~1/10 of what a ViT-S/16 typically needs to clear random from scratch; the CNN inductive bias in Exp 2 is doing all the lifting.
- **Recipe is CNN-tuned, not ViT-tuned.** `Adam`, `lr=1e-4`, no warmup, no weight decay — ViTs usually want long warmup + AdamW + high wd to learn at all from scratch.

**Verdict.** Spatial slot must be a CNN (Track 1) or a *pretrained* ViT (Track 2). No more from-scratch ViT runs.

## Experiment 2 — ResNet18 + Transformer + MLP, from scratch

Second run of the modular pipeline. A CNN replaces the ViT in the spatial slot, still trained from random init (no ImageNet weights). Purpose: measure the CNN vs. ViT gap at this data scale (45k videos) with everything else held constant.

**Config:** `experiment=resnet_transformer_mlp` (`src/configs/experiment/resnet_transformer_mlp.yaml`)

### Components

| Slot | Module | Config |
|---|---|---|
| Spatial | `ResNetEncoder(variant=resnet18, pretrained=False)` | 4 stages, 224² input, adaptive avg-pool → per-frame 512-d → (B, T, 512) |
| Temporal | `TransformerTemporal(in_dim=512, num_heads=8, num_layers=2, dropout=0.1, max_len=64)` | prepend learnable CLS, learnable pos-embed, pre-norm TF blocks (GELU, dim_ff=2048), take CLS → (B, 512) |
| Classifier | `MLPClassifier(in_dim=512, hidden_dim=512, num_classes=33, dropout=0.5)` | Linear(512→512) → GELU → Dropout(0.5) → Linear(512→33) |

### Parameter budget (exact)

| Module | Params |
|---|---|
| ResNet18 | 11.18 M |
| Temporal transformer (2 layers, d=512) | 6.34 M |
| MLP head (512→512→33) | 0.28 M |
| **Total** | **17.80 M** |

### Training setup

- **Data:** `num_frames=4`, `val_ratio=0.2` (internal split from `train/`), ImageNet normalization off (trained from scratch). Train filter applied: the 23 extra shifted-scheme class folders (138 mislabeled clip duplicates) are excluded via a whitelist against `val/` folder names → 44,993 samples.
- **Optim:** Adam, `lr=1e-4`, no weight decay, no scheduler.
- **Batch:** `batch_size=8`, FP32, `num_workers=2`.
- **Epochs:** 20.
- **Hardware:** RTX A5000, 24 GB, CUDA 12.8.
- **Checkpoint:** `/Data/challenge_sb/best_model_resnet_tf_mlp_scratch.pt` (best-by-val-acc).
- **Log:** `/Data/challenge_sb/logs/resnet_transformer_mlp_20260423_194250.log`.
- **Started:** 2026-04-23 19:43.

### Hypothesis

At 45k videos, a from-scratch ResNet18 should learn faster than a from-scratch ViT-S (CNN inductive bias dominates when pretraining is absent). Expected ceiling: **20–30 % val top-1**, likely a few points above the ViT-scratch run. Still below a pretrained baseline, but useful as an ablation anchor for the spatial slot.

### Results

| Epoch | Train loss | Train acc | Val loss | Val acc |
|---|---|---|---|---|
| 1 | 3.3342 | 0.0667 | 3.2680 | 0.0891 |
| 2 | 3.2932 | 0.0791 | 3.2852 | 0.0850 |
| 3 | 3.2676 | 0.0880 | 3.2251 | 0.0981 |
| 4 | 3.2277 | 0.1005 | 3.1835 | 0.1123 |
| 5 | 3.2021 | 0.1076 | 3.1691 | 0.1120 |
| 6 | 3.1751 | 0.1164 | 3.1296 | 0.1278 |
| 7 | 3.1350 | 0.1285 | 3.1202 | 0.1441 |
| 8 | 3.0955 | 0.1417 | 3.0487 | 0.1538 |
| 9 | 3.0344 | 0.1610 | 2.9814 | 0.1784 |
| 10 | 2.9782 | 0.1741 | 2.9380 | 0.1901 |
| 11 | 2.9191 | 0.1901 | 2.8486 | 0.2015 |
| 12 | 2.8703 | 0.2029 | 2.7910 | 0.2297 |
| 13 | 2.8151 | 0.2164 | 2.7536 | 0.2382 |
| 14 | 2.7802 | 0.2245 | 2.6889 | 0.2551 |
| 15 | 2.7220 | 0.2400 | 2.6863 | 0.2587 |
| 16 | 2.6667 | 0.2552 | 2.6456 | 0.2628 |
| 17 | 2.6257 | 0.2624 | 2.5794 | 0.2817 |
| 18 | 2.5751 | 0.2730 | 2.5527 | 0.2921 |
| 19 | 2.5290 | 0.2875 | 2.5290 | 0.2979 |
| 20 | 2.4883 | 0.3000 | 2.4699 | **0.3149** |

- **Best val top-1:** 0.3149 (≈ 10× random-chance 1/33)
- **At epoch:** 20 (last — curve is still climbing)
- **Wall-clock per epoch:** 5.6 min (total: 1.87 h on RTX A5000)
- **Peak VRAM:** not recorded; ResNet18 + B=8 fits comfortably on 24 GB
- **Test-set submission score (if submitted):** _TBD_

#### Observations

- **Still learning at 20 epochs.** Val acc climbed monotonically every epoch (0.089 → 0.315), mirrored by train acc (0.067 → 0.300). Train and val accuracies are essentially tied (no generalization gap yet), so the model is under-fitting, not over-fitting — more epochs would likely help.
- **CNN-vs-ViT-from-scratch, as hypothesised.** 31.5 % beats the ViT-S from-scratch ceiling band (15–25 %), consistent with CNN inductive bias dominating when pretraining is absent at 45 k-video scale.
- **Next levers (highest expected payoff first):** (i) ImageNet-pretrained ResNet18 (Track 2) — expected to add ~15–25 pts of val acc for free; (ii) longer schedule (50 + epochs) with cosine LR decay; (iii) stronger temporal mixing (more frames per clip, or `num_layers=4`); (iv) standard video augmentations.


## Experiment 3 — ResNet18 + DiffTransformer + MLP, from scratch

Third run of the modular pipeline. Same spatial backbone and head as Experiment 2, but the temporal block is new: it explicitly injects **per-frame forward differences** alongside the raw frame features before attention. The hypothesis is that for "What Happens Next?"-style data — where the label is about *motion between frames*, not per-frame content — giving attention direct access to `x[t+1] - x[t]` saves it from having to learn the subtraction internally.

**Config:** `experiment=resnet_difftransformer_mlp` (`src/configs/experiment/resnet_difftransformer_mlp.yaml`)

### Components

| Slot | Module | Config |
|---|---|---|
| Spatial | `ResNetEncoder(variant=resnet18, pretrained=False)` | 4 stages, 224² input, adaptive avg-pool → per-frame 512-d → (B, T, 512) |
| Temporal | `DiffTransformerTemporal(in_dim=512, num_heads=8, num_layers=2, dropout=0.1, max_len=64)` | see below |
| Classifier | `MLPClassifier(in_dim=1024, hidden_dim=512, num_classes=33, dropout=0.5)` | Linear(1024→512) → GELU → Dropout(0.5) → Linear(512→33) |

### Temporal block — how it differs from Experiment 2

Let `x ∈ (B, T, d)` be the stack of frame features (`d=512`, `T=4`).

1. **Forward differences**: `diff[t] = x[t+1] - x[t]` for `t < T-1`, yielding `(B, T-1, d)`.
2. **Learnable end-sentinel**: `diff[T-1]` is a learnable parameter of shape `(1, 1, d)` (trunc-normal init, σ=0.02). This replaces the missing "next frame" at the end of the clip with something the model can tune, rather than a hard-coded zero.
3. **Feature-dim concat**: tokens = `concat([x, diff], dim=-1)` → `(B, T, 2d)`. Each token carries the raw frame feature *and* the change to the next frame.
4. **CLS + learnable positional embedding** at `d_model = 2d = 1024`, then `num_layers=2` pre-norm `TransformerEncoderLayer`s (GELU, `dim_feedforward=4096`).
5. **Output**: CLS token → LayerNorm → `out_dim=1024` (no lossy projection by default; the MLP head consumes the full representation).

Implementation: [`src/models/temporal/diff_transformer.py`](src/models/temporal/diff_transformer.py).

### Parameter budget (exact)

| Module | Params |
|---|---|
| ResNet18 | 11.18 M |
| DiffTransformer (2 layers, d_model=1024, end-diff + CLS + pos-embed) | ~25.2 M |
| MLP head (1024→512→33) | 0.54 M |
| **Total** | **~36.98 M** |

Confirmed by `sum(p.numel() for p in m.parameters())` on a smoke-built model.

### Training setup

- **Data:** `num_frames=4`, `val_ratio=0.2` (internal split from `train/`), ImageNet normalization off (trained from scratch). Train filter applied: class folders restricted to the whitelist derived from `val/` (drops the 23 mislabeled-scheme folders, same as Experiment 2).
- **Optim:** Adam, `lr=1e-4`, no weight decay, no scheduler.
- **Batch:** `batch_size=8`, FP32, `num_workers=2`.
- **Epochs:** 20.
- **Hardware:** local CUDA GPU, CUDA 12.8.
- **Checkpoint:** `/Data/challenge_sb/checkpoints/diff_transformer_best.pt` (best-by-val-acc).
- **Log:** `/Data/challenge_sb/logs/diff_transformer_20ep_20260423-203421.log`.
- **Started:** 2026-04-23 20:34.
- **Live progress:** `bash scripts/progress.sh /Data/challenge_sb/logs/diff_transformer_20ep_20260423-203421.log 20` (run in a separate terminal).

### Hypothesis

Versus Experiment 2 (same spatial + head, vanilla Transformer), the diff block adds ~7 M params but gives attention a hand-computed motion signal. Two plausible outcomes:

- **Lift (expected):** at this data scale, explicit differences are a useful inductive bias for motion-centric labels → a few points of val top-1 above Experiment 2's final.
- **Neutral / slight regression:** a plain self-attention with enough layers/data can recover the same representation internally; at 45k videos the 2× token-dim doubling of the FFN could just overfit faster.

Target: **beat Experiment 2's val top-1 by ≥ 2 points** at the same epoch budget. Secondary: confirm the learnable end-sentinel stays finite (no divergence on the sentinel token).

### Results

*To be filled in after the run completes.*

| Epoch | Train loss | Train acc | Val loss | Val acc | Notes |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| … |  |  |  |  |  |
| 20 |  |  |  |  |  |

- **Best val top-1:** _TBD_
- **At epoch:** _TBD_
- **Wall-clock per epoch:** _TBD_ min (total: _TBD_ h)
- **Peak VRAM:** _TBD_ GB
- **Test-set submission score (if submitted):** _TBD_

#### Observations

_TBD — whether the explicit diffs yield a measurable lift over Experiment 2 or whether the model absorbs the same signal without help; whether overfitting onset shifts earlier/later given the extra FFN width; behavior of the learnable end-sentinel._

## Experiment 4 — CMT (Channel-Motion Transformer), ResNet18 from scratch

Fourth run. **Standalone architecture, not modular** — it breaks the `(B, T, d)` contract because the per-frame CNN keeps its spatial map, and the temporal/pooling stack runs *per feature-channel* before fusing across channels. The idea: treat each CNN channel's `(T, H', W')` activation map as its own "motion stream" that tracks where that channel fires across time, encode each stream with a shared space-time sub-net, and pool the resulting channel-vectors with a Set-Transformer-style attention.

**Config:** `experiment=cmt_from_scratch` (`src/configs/experiment/cmt_from_scratch.yaml`)

### Pipeline

```
video (B, T, 3, 224, 224)
 └─ ResNet-18 trunk (shared over T, no avgpool/fc) → (B, T, 512, 7, 7)
 └─ 1×1 Conv2d 512→C'=128 (shared over T)         → (B, T, 128, 7, 7)
 └─ permute + learned temporal pos-embed (1,1,T,1,1) → (B, 128, T, 7, 7)
 └─ reshape → (B·128, 1, T, 7, 7)
 └─ PerChannelMotionNet (shared weights over C')  → (B·128, 512)
 └─ reshape → (B, 128, 512)  # set of 128 motion tokens
 └─ 4× TransformerEncoder (d=512, 8 heads, FFN 2048, pre-norm)
 └─ PMA (1 learned seed query, cross-attn)        → (B, 512)
 └─ MLP head 512→1024→33                          → (B, 33)
```

`PerChannelMotionNet` is a (2+1)D ResNet-3D-lite:

```
Conv3d(1→96, k=3) + GN + GELU
Stage1: 2× STBlock(96→96)                 # (2+1)D block: Conv(1,3,3)+GN+GELU + Conv(3,1,1)+GN + residual + GELU
MaxPool3d((1,2,2))                        # 7×7 → 3×3 spatially, T unchanged
Stage2: STBlock(96→192), STBlock(192→192)
MaxPool3d((1,2,2))                        # 3×3 → 1×1 spatially
Stage3: STBlock(192→384), STBlock(384→384)
GlobalAvgPool3d + Linear(384→512)
```

### Components

| Slot | Module | Config |
|---|---|---|
| Frame CNN | `ResNet18Trunk(pretrained=False)` | truncated before avgpool/fc; out = (512, 7, 7) |
| Bottleneck | `Conv2d(512 → 128, k=1)` | shared over T; defines "motion channels" C' |
| Temporal pos-embed | learnable `(1, 1, T, 1, 1)` | added to each channel's time axis |
| Per-channel motion net | `PerChannelMotionNet(d=512, widths=(96,192,384))` | shared over C', applied as `(B·C', 1, T, 7, 7) → (B·C', 512)` |
| Set encoder | 4× `nn.TransformerEncoderLayer(d=512, h=8, FFN=2048, dropout=0.1, pre-norm)` | permutation-equivariant mixing over C'=128 tokens |
| Set pool | `PMA(d=512, heads=8, FFN=2048)` | 1 learned seed query → one 512-d vector |
| Head | `Linear(512→1024) → GELU → Dropout(0.1) → Linear(1024→33)` | |

### Parameter budget (exact)

| Module | Params |
|---|---|
| ResNet-18 trunk | 11.18 M |
| 1×1 bottleneck | 0.07 M |
| Per-channel motion net | 4.11 M |
| Set encoder (4 layers) | 12.61 M |
| PMA set pool | 3.15 M |
| MLP head | 0.56 M |
| **Total** | **31.68 M** |

### Training setup

- **Data:** `num_frames=8` (doubled vs. Exps 1–3 — the per-channel motion net needs real temporal extent to be worth the compute), `val_ratio=0.2` internal split, **no ImageNet normalization** (from-scratch), same train-filter whitelist against `val/` classes (→ 44,993 samples).
- **Optim:** Adam, `lr=3e-4`, no weight decay, no scheduler.
- **Batch:** `batch_size=16`, FP32, `num_workers=2`.
- **Epochs:** 20.
- **Hardware:** RTX A5000, 24 GB, CUDA 12.8.
- **Checkpoint:** `/Data/challenge_sb/cmt_best.pt` (best-by-val-acc).
- **Log:** `/Data/challenge_sb/logs/cmt_20260423_211343.log`.
- **Started:** 2026-04-23 21:13.

### Hypothesis

Where Experiment 3 (DiffTransformer) injects motion as an *explicit first-difference on frame features*, CMT decomposes motion *per CNN-channel* over the full `(T, H, W)` activation volume, then attends over the resulting motion descriptors. The novel claim: *for a fixed channel, watching the (H,W) activation move across T should encode object-/body-/hand-motion patterns that a mean-pool (Exp 2) or a single-vector frame-diff (Exp 3) can't easily recover.* Expected ceiling from scratch at this data scale: **25–35 % val top-1** — on par with Exps 2–3 if the novel block is doing nothing useful, and a few points above if the per-channel motion signal genuinely helps. A win here would come *primarily from the motion block*, since the backbone and data are a weaker setting than Exps 2–3 (more params at 31.7 M vs. 17.8 M / ~37.0 M, same optimizer, no pretraining, doubled T).

Key ablations to run later if this works:

1. Replace `PerChannelMotionNet` with mean-over-T + a small MLP (kills the motion claim → isolates whether the per-channel space-time block is doing the work).
2. Drop the 1×1 bottleneck (C' = 512 → would it help or hurt?).
3. Swap 4-layer set encoder for 1 layer + PMA only (isolates set-mixing depth).

### Results — failed

**Val top-1 plateaued at ~7 %** (random-chance floor is 1/33 ≈ 3 %). 18–28 pts below the 25–35 % hypothesis, and flat from early epochs.

#### Diagnosis

- **Bottleneck kills the signal before motion sees it.** The 1×1 `512→128` squeeze runs *before* any temporal block. From scratch the trunk's 512 channels aren't meaningfully distinct yet, so the 128 "motion streams" are mostly noise → `PerChannelMotionNet` has nothing to encode.
- **Shared-weight per-channel net assumes a trained backbone.** Weight-sharing across 128 streams only pays off if each channel already carries a distinct semantic — bootstrapping problem at scratch.
- **Recipe too hot for 31.7 M params.** `Adam`, `lr=3e-4`, no warmup, no wd, GroupNorm-heavy 3D stack → early collapse is the expected failure mode.

**Verdict.** Architecture isn't clearly broken, but the from-scratch setting is the wrong harness for it. Revisit only with a pretrained ResNet trunk + AdamW + warmup before declaring CMT dead.

#### Common thread (Exp 1 + Exp 4)

45k clips is too small to train any high-capacity model from scratch with this optimizer recipe. Only Exp 2 (smallest, simplest) actually learned — and Exp 5 showed even it overfits past ep 18. Track 1 needs **regularization + early stopping**, not more architecture.

## Experiment 5 — ResNet18 + Transformer + MLP, from scratch (overnight, tuned)

Same architecture as Experiment 2 (ResNet+TF+MLP from scratch), but the training recipe is overhauled for a ~12 h budget. Training loop patched (`src/train.py`) with **AMP (fp16)**, **AdamW + weight decay**, **linear warmup + cosine LR schedule**, and **gradient clipping**. Clip length doubled to 8 frames.

**Config:** `experiment=resnet_transformer_mlp` (+ CLI overrides, see below)

### Components

Unchanged from Experiment 2 except `T=8` (instead of 4):

| Slot | Module |
|---|---|
| Spatial | `ResNetEncoder(resnet18, pretrained=False)` → (B, 8, 512) |
| Temporal | `TransformerTemporal(in_dim=512, heads=8, layers=2, dropout=0.1)` → (B, 512) |
| Classifier | `MLPClassifier(512→512→33, dropout=0.5)` → (B, 33) |

Total: **17.80 M params** (identical).

### Training setup

| Param | Value | Note |
|---|---|---|
| Epochs | 110 | ≈ 10.7 h @ ~5.9 min/epoch |
| Batch size | 16 | Up from 8 |
| `num_frames` | 8 | Up from 4 (2× temporal context) |
| Optimizer | **AdamW**, `weight_decay=0.05` | Was Adam, wd=0 |
| LR | peak **3e-4**, linear warmup 5 epochs, cosine to 0 | Was constant 1e-4 |
| Precision | **fp16 AMP** | 1.5-1.8× speedup |
| Grad clip | **1.0** | L2 norm |
| Data augmentation | unchanged (resize 224, hflip, normalize) |  |
| `num_workers` | 8 | Up from 2 |
| Hardware | RTX A5000, 24 GB | 100 % util, ~7 GB VRAM |
| Checkpoint | `/Data/challenge_sb/best_model_resnet_tf_mlp_overnight.pt` |  |
| Log | `/Data/challenge_sb/logs/resnet_tf_mlp_overnight_20260423_214406.log` |  |
| Started | 2026-04-23 21:44 |  |

### Hypothesis

Experiment 2 hit **31.5 % val top-1** in 20 epochs with the curve still linearly climbing, so the bottleneck was the training schedule, not the model. Doubling the frames, adding a proper LR schedule, and running 5.5× longer should push val top-1 well into the **45-52 %** band, with the cosine tail absorbing late-stage optimization. Weight decay + dropout are expected to contain late-epoch overfitting despite the long run.


### Results

| Epoch | Train loss | Train acc | Val loss | Val acc |
|---|---|---|---|---|
| 1 | 3.33 | 0.085 | 3.19 | 0.105 |
| 5 | 2.85 | 0.209 | 2.93 | 0.196 |
| 10 | 2.43 | 0.316 | 2.53 | 0.294 |
| 15 | 2.03 | 0.414 | 2.36 | 0.348 |
| 18 | 1.78 | 0.477 | 2.40 | 0.352 |
| 20 | 1.58 | 0.528 | 2.47 | 0.347 |
| 30 | 0.68 | 0.780 | 3.28 | 0.329 |
| 40 | 0.346 | 0.892 | 4.06 | 0.321 |
| 50 | 0.225 | 0.933 | 4.52 | 0.329 |
| 60 | 0.150 | 0.959 | 5.22 | 0.330 |
| 70 | 0.086 | 0.979 | 6.57 | 0.332 |
| 80 | 0.040 | 0.991 | 7.19 | 0.339 |
| 85 | 0.029 | 0.994 | 7.15 | **0.358** (first new best since ep 18) |
| 90 | 0.016 | 0.997 | 7.34 | 0.360 |
| 95 | 0.011 | 0.998 | 7.24 | 0.364 |
| 100 | 0.006 | 0.999 | 7.24 | 0.369 |
| **102** | **0.005** | **0.999** | **7.24** | **0.3725** ← best checkpoint |
| 110 | 0.004 | 0.999 | 7.16 | 0.372 |

- **Best val top-1 (internal train-split):** 0.3725 @ epoch 102
- **Held-out eval (`evaluate.py` on `processed_data/val/`, 6745 samples):** **top-1 0.2302 / top-5 0.4903**
- **Wall-clock per epoch:** 5.76 min — total 10 h 33 min on RTX A5000 (21:44 → 08:17)
- **Peak VRAM:** ~7 GB (well under the 24 GB budget)
- **Test-set submission score:** _not yet submitted_

#### Observations

- **Big gap between "val" numbers.** Training reports 0.3725 on an internal 20 % slice of `train/` (same distribution as training), but the held-out `val/` folder gives 0.2302 — a **14-point drop**. The model memorized the training distribution; the train-acc trajectory (0.999 by epoch 50) and val loss climbing to 7+ confirm it. The internal-split number is the wrong yardstick. Future experiments should report `val/` top-1 as the primary metric, or split val off from `val/`.
- **The recipe actually hurt.** Experiment 2 hit 0.3149 val in 20 epochs with `T=4`, `lr=1e-4`, constant LR, no AMP — still climbing. Here with `T=8`, warmup + cosine to 3e-4, AdamW wd=0.05, 110 epochs, we got 0.3725 on the same (internal) metric — **only +6 points for 5.5× the compute**, and the cosine tail is mostly redoing what dropped off between ep 20-85. On held-out `val/` Exp 5's 0.2302 probably trails Exp 2's (if we re-ran it with an honest val eval). The hypothesis that "schedule was the bottleneck" was wrong: the bottleneck was **regularization / data**, not schedule.
- **Mid-training dip.** Val acc peaked at 0.352 around epoch 18, then fell to ~0.32 and stayed flat through epoch 80 before the cosine tail recovered to 0.372. That dip coincides with train-acc blowing past 0.5 — i.e. the LR at 3e-4 was too high once the model started fitting noise, and cosine decay only saved it at the end. A shorter cosine (30-40 ep) or early-stopping at ep 18-20 would likely match this final number at a fraction of the cost.
- **AMP / batch 16 was effective.** 5.76 min/epoch at T=8 vs Exp 2's 5.6 min/epoch at T=4 — essentially free 2× temporal context.
- **Next levers.** (i) Real regularization — label smoothing, stronger augmentation (RandAugment / mixup / cutmix / temporal jitter), not just dropout. (ii) Drop LR to 1e-4 and keep the cosine. (iii) Early-stop at the first val plateau instead of running 110 epochs. (iv) Evaluate all checkpoints going forward on held-out `val/`, not an internal split, so we're optimizing the right number.
